# SFT Validator — Spec v4 (Structural + Semantic only)

**Stage 1 — Structural Integrity:** 1.1 Schema → 1.3 Garbage symbols → 1.2 Exact dedup 
**Stage 2 — Semantic Dedup:** `user + assistant`, `BAAI/bge-m3` chunks of 1000, L2-norm, block by `(subject, class)`, `cos > 0.88` → `semantic_dupe`.

Outputs: `clean + unified rejected + report`. 

## 0. Setup

In [ ]:
!nvidia-smi 2>&1 | head -5
!pip install -q FlagEmbedding sentence-transformers pandas numpy matplotlib
import sys
print(sys.version)

## 1. Load 

In [ ]:
import json, os, glob
from pathlib import Path
OUT_DIR = Path('/content/reports_v4')
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = os.environ.get('DATA_PATH', '')
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('IN_COLAB:', IN_COLAB)
if IN_COLAB and (not DATA_PATH or not os.path.exists(DATA_PATH)):
    print('Upload your .jsonl:')
    uploaded = files.upload()
    if uploaded:
        DATA_PATH = f"/content/{list(uploaded.keys())[0]}"
if not DATA_PATH or not os.path.exists(DATA_PATH):
    cands = sorted(glob.glob('/content/*.jsonl')) + sorted(glob.glob('*.jsonl'))
    DATA_PATH = cands[0] if cands else DATA_PATH
print('DATA_PATH:', DATA_PATH)
objs, pre_quarantined, src_note = [], [], DATA_PATH
if DATA_PATH and os.path.exists(DATA_PATH):
    with open(DATA_PATH, encoding='utf-8') as f:
        for ln, line in enumerate(f):
            if not line.strip(): continue
            try:
                objs.append((ln, json.loads(line)))
            except Exception as e:
                pre_quarantined.append({'id': ln, 'stage': 'stage1_integrity',
                    'reasons': [f'schema: json_decode_error: {e}'], 'warns': [],
                    'meta': None, 'messages': None})
    print(f'parsed rows: {len(objs)} | bad-json quarantined: {len(pre_quarantined)}')
else:
    print('No file found — upload above.')

## 2. Utils

In [ ]:
import re, unicodedata, hashlib
from collections import Counter, defaultdict
BN_DIGITS = str.maketrans('০১২৩৪৫৬৭৮৯', '0123456789')
def normalize_digits(s): return s.translate(BN_DIGITS)
def bn_tokens(s):
    s = normalize_digits(s.lower())
    return re.findall(r'[\u0980-\u09ff]+|[a-z0-9]+', s)
def hash_normalize(s):
    s = unicodedata.normalize('NFKC', s)
    s = normalize_digits(s.lower())
    s = re.sub(r'[^\u0980-\u09ff\w\s]', '', s)
    return re.sub(r'\s+', ' ', s).strip()
def get_field(block, labels):
    for lab in labels:
        m = re.search(re.escape(lab) + r'\s*:\s*([^\n]+)', block)
        if m: return m.group(1).strip()
    return 'UNKNOWN'
def parse_meta(sys_content):
    i = sys_content.find('পাঠ্যসূচি বিবরণী')
    block = sys_content[i:i+900] if i >= 0 else sys_content[-900:]
    sj = get_field(block, ['- বিষয়', '- বিষয়'])
    kl = get_field(block, ['- শ্রেণি', '- শ্রেণী'])
    return {'subject': sj, 'subject_norm': sj.strip().lower(), 'class': kl.strip()}
print('utils loaded')

## 3. Stage 1 — Structural Integrity (1.1 schema → 1.3 garbage → 1.2 exact dupe)

In [ ]:
COS_THRESH = 0.88
GARBLE_RE = re.compile(r'[£€¥©®]{1,}')
DANDA_RE = re.compile(r'[।॥]+')
MIXED_RE = re.compile(r'[\u0980-\u09FF][\u0900-\u0939\u093E-\u094D\u0950-\u0954\u0960-\u0963]|[\u0900-\u0939\u093E-\u094D\u0950-\u0954\u0960-\u0963][\u0980-\u09FF]')
# Guard: markdown fences (```) and rule-only lines (runs of _ - = | + spaces,
# e.g. long-division bars '_______' / '-----') are layout, never garble.
FENCE_MARK_RE = re.compile(r'(?m)^\s*```.*$')
RULE_LINE_RE = re.compile(r'(?m)^[\s_\-=|]+$')
def strip_structural(s):
    s = FENCE_MARK_RE.sub(' ', s)
    return RULE_LINE_RE.sub(' ', s)
def has_mixed(s): return bool(MIXED_RE.search(DANDA_RE.sub('', s)))
def validate_record(obj, seen):
    # 1.1 Schema Check
    msgs = (obj.get('messages') or []) if isinstance(obj, dict) else []
    if not isinstance(obj, dict) or 'messages' not in obj:
        return False, ['schema: root object missing messages array'], None
    if len(msgs) < 3:
        return False, ['schema: messages needs at least 3 items'], None
    if [m.get('role') for m in msgs[:3]] != ['system', 'user', 'assistant']:
        return False, ['schema: first 3 roles must be [system, user, assistant]'], None
    usr_c = msgs[1].get('content') or ''
    ast_c = msgs[2].get('content') or ''
    if not usr_c.strip():
        return False, ['schema: empty user content'], None
    if not ast_c.strip():
        return False, ['schema: empty assistant content'], None
    meta = parse_meta(msgs[0].get('content') or '')
    # 1.3 Garbage Symbols (structural layout stripped first)
    usr_g, ast_g = strip_structural(usr_c), strip_structural(ast_c)
    if GARBLE_RE.search(usr_g) or GARBLE_RE.search(ast_g):
        return False, ['garbled: currency/copyright noise'], {'meta': meta}
    if has_mixed(usr_g) or has_mixed(ast_g):
        return False, ['garbled: mixed_script Bengali+Devanagari'], {'meta': meta}
    # 1.2 Exact Deduplication (user prompt only; hash added only on full pass)
    h = hashlib.sha256(hash_normalize(usr_c).encode('utf-8')).hexdigest()[:16]
    if h in seen:
        return False, ['exact_dupe'], {'meta': meta, 'hash': h}
    seen.add(h)
    return True, [], {'meta': meta, 'hash': h}
print('Stage-1 defined')

### 3b. Run Stage-1

In [ ]:
seen, passed, quarantined = set(), [], list(pre_quarantined)
for ln, o in objs:
    ok, reasons, info = validate_record(o, seen)
    rec = {'id': ln, 'reasons': reasons, 'warns': [],
           'meta': (info or {}).get('meta'), 'obj': o}
    (passed if ok else quarantined).append(rec)
print(f'total {len(objs) + len(pre_quarantined)} | PASS {len(passed)} | QUARANTINE {len(quarantined)}')
for r in quarantined[:15]: print(f"- id {r['id']} -> {r['reasons']}")

## 4. Stage 2 — Semantic Dedup (bge-m3, chunks of 1000, block by subject+class)

In [ ]:
if 'passed' not in globals():
    raise RuntimeError("Run cell 3b first — 'passed' not found.")
import numpy as np
EMB_BATCH, EMB_CHUNK = 32, 1000
texts = [(r['obj']['messages'][1]['content'] + ' \n ' + r['obj']['messages'][2]['content'])[:2000] for r in passed]
print(f'to-embed: {len(texts)} (chunks of {EMB_CHUNK})')
embs = None
if texts:
    try:
        from FlagEmbedding import BGEM3FlagModel
        bge = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
        parts = []
        for s in range(0, len(texts), EMB_CHUNK):
            out = bge.encode(texts[s:s+EMB_CHUNK], batch_size=EMB_BATCH, max_length=1024, return_dense=True)
            e = np.array(out['dense_vecs'], dtype=np.float32)
            e = e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-12)
            parts.append(e)
            print(f'  chunk {s}-{s+len(e)} done')
        import numpy as _np
        embs = parts[0] if len(parts) == 1 else _np.concatenate(parts, axis=0)
        print('bge-m3:', embs.shape)
    except Exception as e:
        print('bge-m3 failed, Jaccard fallback:', repr(e)[:400])
        embs = None
def block_key(r):
    m = r['meta'] or {}
    return (str(m.get('subject_norm', '?'))[:30], str(m.get('class', '?'))[:20])
blocks = defaultdict(list)
for idx, r in enumerate(passed): blocks[block_key(r)].append(idx)
print('blocks:', len(blocks))
def jacc(a, b):
    sa, sb = set(bn_tokens(a)), set(bn_tokens(b))
    return len(sa & sb) / max(1, len(sa | sb))
keep, dropped = [], []
for bk, idxs in blocks.items():
    kept = []
    for j in idxs:
        dup = False
        for k in kept:
            sim = float(embs[j] @ embs[k]) if embs is not None else jacc(texts[j], texts[k])
            if sim > COS_THRESH:
                pj, pk = passed[j], passed[k]
                dropped.append({'id': pj['id'], 'stage': 'semantic_dupe',
                    'reasons': [f'semantic_dupe sim={sim:.3f} vs id {pk["id"]}'],
                    'warns': [], 'meta': pj['meta'], 'messages': pj['obj']['messages']})
                dup = True
                break
        if not dup:
            kept.append(j)
            keep.append(passed[j])
print(f'keep {len(keep)} | dropped {len(dropped)}')

## 5. Save: clean + rejected (unified) + report

In [ ]:
clean_path = OUT_DIR / 'specv4-clean.jsonl'
rej_path = OUT_DIR / 'specv4-rejected.jsonl'
quar_path = OUT_DIR / 'specv4-quarantine.jsonl'
sem_path = OUT_DIR / 'specv4-semdropped.jsonl'
report_path = OUT_DIR / 'report.json'
with open(clean_path, 'w', encoding='utf-8') as f:
    [f.write(json.dumps(r['obj'], ensure_ascii=False) + '\n') for r in keep]
with open(quar_path, 'w', encoding='utf-8') as f:
    [f.write(json.dumps({'id': r['id'], 'stage': 'stage1_integrity', 'reasons': r['reasons'], 'warns': r['warns'], 'meta': r['meta'], 'messages': (r['obj'].get('messages') if isinstance(r.get('obj'), dict) else None)}, ensure_ascii=False) + '\n') for r in quarantined]
with open(sem_path, 'w', encoding='utf-8') as f:
    [f.write(json.dumps(d, ensure_ascii=False) + '\n') for d in dropped]
rejected = [{'id': r['id'], 'stage': 'stage1_integrity', 'reasons': r['reasons'], 'warns': r['warns'], 'meta': r['meta'], 'messages': (r['obj'].get('messages') if isinstance(r.get('obj'), dict) else None)} for r in quarantined] + dropped
rejected.sort(key=lambda x: x['id'])
with open(rej_path, 'w', encoding='utf-8') as f:
    [f.write(json.dumps(r, ensure_ascii=False) + '\n') for r in rejected]
total = len(keep) + len(rejected)
report = {'source': src_note if 'src_note' in dir() else DATA_PATH, 'total': total,
    'stage1_pass': len(passed), 'stage1_quarantine': len(quarantined),
    'semantic_dropped': len(dropped), 'final_clean': len(keep), 'rejected_total': len(rejected),
    'stage1_pass_rate': round(len(passed) / max(1, total), 4),
    'final_pass_rate': round(len(keep) / max(1, total), 4),
    'quarantine_top_reasons': Counter(sum([r['reasons'] for r in quarantined], [])).most_common(10)}
open(report_path, 'w', encoding='utf-8').write(json.dumps(report, ensure_ascii=False, indent=2))
print(json.dumps(report, ensure_ascii=False, indent=2))
print(f'\nwrote: {clean_path} | {rej_path} | {report_path}')
assert len(keep) + len(rejected) == total
print(f'check: clean({len(keep)}) + rejected({len(rejected)}) == total({total}) OK')
if IN_COLAB:
    try:
        from google.colab import files as _f
        _f.download(str(clean_path)); _f.download(str(rej_path)); _f.download(str(report_path))
    except Exception as e: print('download skipped:', e)